# 📄 Informe Técnico Completo: Multimodal Information Retrieval System (RAG)

Este informe detalla exhaustivamente la arquitectura, el diseño del pipeline, el corpus utilizado, los resultados empíricos (métricas) y las implementaciones avanzadas (funcionalidades extra) del sistema de recuperación de información multimodal. 

Además, se adjuntan fragmentos de código clave y las **justificaciones arquitectónicas** de por qué se eligieron ciertas tecnologías frente a sus alternativas.

## 1. Corpus y Preprocesamiento

- **Tamaño del Dataset:** ~700 productos reales (extraídos del dataset público de *Amazon Reviews 2023* en Hugging Face).
- **Categorías (10):** Musical Instruments, Video Games, Pet Supplies, Cameras, Electronics, Sports & Outdoors, Toys, Home & Kitchen, Cell Phones, Office Products.
- **Estructura del Dato:** Cada documento consta de Título, Categoría, Descripción textual y el path local a su Imagen representativa descargada.
- **Estrategia de Indexación:** Se concatenaron los campos textuales clave (`Categoría` + `Título` + `Descripción`) para extraer la riqueza semántica completa del producto.


## 2. Arquitectura del Sistema: Decisiones de Diseño

### ¿Por qué CLIP en lugar de BLIP?
En sistemas multimodales, a menudo se debate entre usar CLIP (OpenAI) o BLIP (Salesforce). 
Se eligió **CLIP (Contrastive Language-Image Pretraining)** porque está diseñado específicamente para generar **embeddings contrastivos**. Es decir, proyecta imágenes y textos en el *mismo espacio latente* optimizando para que la distancia entre una imagen y su descripción sea mínima. Esto es ideal para sistemas de **Recuperación (Retrieval)** basados en bases de datos vectoriales (k-NN).
Por otro lado, **BLIP** es un modelo más enfocado en la **Generación** (Visual Question Answering, Image Captioning). Para buscar rápidamente entre miles de documentos usando matemática vectorial pura (Similitud Coseno), CLIP es computacionalmente superior y más directo.

### ¿Por qué ChromaDB en lugar de FAISS?
Aunque FAISS (Facebook) es ultrarrápido para búsquedas en memoria, **carece de una gestión robusta de metadatos**. Si usáramos FAISS, tendríamos que mantener una base SQLite aparte para los textos y títulos, haciendo inner joins manualmente. **ChromaDB** soluciona esto actuando como un motor híbrido: maneja un índice vectorial en grafo HNSW para búsquedas veloces, y almacena la payload (texto/metadatos) usando SQLite/Parquet integradamente, con persistencia nativa en disco en una sola línea de código.

### 💻 Código Interno: Normalización de Embeddings para Similitud Rápida
Para acelerar al máximo ChromaDB, en `src/embeddings.py` aplicamos una normalización L2 inmediatamente después de que CLIP genera el vector. Al tener todos los vectores magnitud 1.0, el costoso cálculo de Distancia Coseno se simplifica a un simple Producto Punto (Dot Product) a nivel de CPU.

In [ ]:
import torch

# Fragmento de src/embeddings.py (Clase CLIPEmbedder)
@torch.no_grad()
def embed_text(self, texts):
    text_input = self.tokenizer(texts).to(self.device)
    text_features = self.model.encode_text(text_input)
    
    # MÁGIA AQUÍ: Normalización L2 para facilitar Similitud Coseno -> Producto Punto
    text_features /= text_features.norm(dim=-1, keepdim=True)
    
    return text_features.cpu().numpy()

## 3. Pipeline de Recuperación Multimodal

- **Texto:** Se vectoriza con CLIP Text Encoder.
- **Voz:** Se transcribe usando la API nativa de Google (`gemini-2.5-flash` con MIME `audio/wav`) y luego se trata como texto.
- **Imagen:** La foto pasa por el ViT de CLIP para obtener el vector visual y se busca directamente contra el espacio latente del corpus.

### 💻 Código Interno: RAG - Formateo de Contexto
Una vez ChromaDB devuelve los candidatos, la clase `RAGGenerator` prepara el prompt inyectando los documentos textuales como contexto para Gemini.

In [ ]:
# Fragmento de src/generation.py (Clase RAGGenerator)
def format_context(self, retrieved_docs):
    if not retrieved_docs:
        return "No se encontraron productos relevantes."
        
    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        title = doc.get('title', 'Producto sin nombre')
        text = doc.get('text', '')
        context_parts.append(f"Producto {i}:\n- Título: {title}\n- Descripción: {text}")
        
    return "\n\n".join(context_parts)

## 4. Métricas y Evaluación del Desempeño

Se anotaron manualmente 20 queries de prueba (Qrels) y se evaluó el pipeline base (CLIP puro):

| Métrica | @1 | @3 | @5 | @10 |
| :--- | :--- | :--- | :--- | :--- |
| **Precision** | 0.6000 | 0.3167 | 0.2700 | 0.1550 |
| **Recall** | 0.3392 | 0.4883 | 0.6650 | 0.7750 |
| **NDCG** | 0.6000 | 0.5211 | 0.5937 | 0.6361 |

**Conclusión Arquitectónica a partir de los datos:**
El sistema presenta un alto **Recall@10 (77.5%)** pero una **Precision@3 baja (31.6%)**. Esto significa que CLIP es fenomenal como motor de *Recall* rápido (trae casi todo lo relevante en la bolsa de los top-10 o top-20), pero no es tan bueno ordenando el *Top-3* exacto. 
Esta observación estadística fue la justificación técnica para implementar la primera funcionalidad extra: un **Re-ranker (Cross-Encoder)**.

## 5. Funcionalidades de Excelencia (+60 pts Extras)

Para mitigar las falencias descubiertas en las métricas y mejorar dramáticamente la UX, se implementaron 4 mecanismos avanzados:

### A. Re-ranking con Cross-Encoder (+15 pts)
Para solucionar la baja precisión Top-3, pasamos de un pipeline puramente *Bi-Encoder* (CLIP) a uno de **Dos Etapas**.
- **Stage 1 (Bi-Encoder):** CLIP devuelve el Top-20 (Máximo Recall).
- **Stage 2 (Cross-Encoder):** Se utiliza `ms-marco-MiniLM-L-6-v2`. A diferencia de CLIP, el Cross-Encoder evalúa la query y el documento concatenados al mismo tiempo, utilizando atención profunda entre las palabras. Es más lento, por eso solo se aplica al Top-20, devolviendo los Top-3 re-ordenados con precisión milimétrica.

### B. Query Expansion con LLM (+15 pts)
Para atacar el problema de *Vocabulary Mismatch* (el usuario busca "guitarra para novatos", el corpus dice "beginner acoustic guitar"), usamos **Pseudo-Relevance Expansion**.
Antes de la búsqueda, Gemini genera 3 reformulaciones de la query. El Retriever (en `src/retrieval.py`) busca el top-15 para cada una de las 4 queries, unifica un megabowl de ~60 candidatos (deduplicando IDs) y luego el Cross-Encoder re-ordena todo. Esto masifica enormemente el espacio de búsqueda latente.

### C. Relevance Feedback - Rocchio Score-Based (+15 pts)
Se incorporaron botones 👍/👎 en los productos devueltos en la interfaz.
- El algoritmo clásico de Rocchio ajusta el vector de la query. Nosotros implementamos un **Rocchio basado en Scores** para aplicarlo post-Cross-Encoder.
- **Fórmula:** `boost = 1.0 + 0.3 * (likes - dislikes) / (total + 1)`
- Si un producto tiene alta aprobación comunitaria, su score semántico recibe un multiplicador (hasta ~1.3x) asegurando que aparezca primero en futuras búsquedas similares.


In [ ]:
# Fragmento de src/relevance_feedback.py
def get_boost_factor(self, doc_id: str) -> float:
    """ Rocchio score-based simplificado (alpha=0.3) """
    if doc_id not in self._data["feedbacks"]:
        return 1.0

    fb = self._data["feedbacks"][doc_id]
    likes, dislikes = fb["likes"], fb["dislikes"]
    total = likes + dislikes
    if total == 0:
        return 1.0

    alpha = 0.3 
    net_score = (likes - dislikes) / (total + 1)
    return 1.0 + alpha * net_score # Devuelve factor multiplicativo

### D. Memoria Conversacional (+15 pts)
Para resolver referencias anafóricas (ej. "¿y de ese producto tienes algo más barato?"), el sistema debe recordar el contexto.
Implementamos una *ventana deslizante* estática que inyecta en el prompt base de Gemini el histórico de la charla (los últimos 6 turnos de `st.session_state`). El LLM ahora realiza **Context-Aware Generation**, pudiendo hilar la conversación orgánicamente como un verdadero vendedor humano.

In [ ]:
# Fragmento de src/generation.py - Inyección de memoria en el Prompt
prompt_template = """
Eres un asistente experto de compras con memoria conversacional.
MEMORIA: Tienes acceso al historial. Úsalo para entender referencias como "ese producto".

--- Historial de Conversación (ultimos turnos) ---
{chat_history}
--- Fin del Historial ---

Contexto Recuperado: {context}
Pregunta Actual: {question}
"""